# Read-once encoder: T4 smoke test

Settings: **Accelerator = GPU T4 x2**, **Internet = on**. Then *Run all*.

Writes `/kaggle/working/smoke_results.json`. Each test is independent; results are saved after every test.

In [ ]:
!nvidia-smi
%pip install -q transformers==5.17.0 tokenizers==0.23.2 huggingface_hub==1.32.0 safetensors==0.8.0

In [ ]:
%%writefile readonce.py
"""Read-once, answer-many packing for a ModernBERT-style encoder (prototype).

Packed sequence (one row per state):

    [CLS] state [SEP] | Q1 block | Q2 block | ... | QK block

    Q block = "choice question: <text>" [SEP] [MASK] opt_0 [MASK] opt_1 ... [SEP]

Attention (built here, passed to the stock Hugging Face ModernBertModel as a per-layer-type mask dict):
    * state tokens attend only to state tokens (the state encoding is question-agnostic);
    * tokens of block i attend to all state tokens and to block i;
    * no attention between different blocks.
Position ids: the state uses 0..S-1 and every block restarts at S ("parallel positions"), so each
question sees the state exactly as if it were the only question.

ModernBERT's local layers use a sliding window. The stock mask computes it on *sequence index*
(|q_idx - kv_idx| <= 64); with parallel positions the window must be computed on *position id*
instead, otherwise block 2..K would be "far" from the end of the state and see less of it than
block 1 does. `build_masks(window_on="position")` does that.

Baseline ("Laya-style" cross-encoder): one row per question, [CLS] Q block state [SEP], full
bidirectional attention, so the state is re-encoded for every question (and is question-aware).
Both models share the same scorer: an MLP over the hidden state at each [MASK] marker, softmax
over that question's options. Yes/no questions are 2-option choice questions.
"""
from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F


# ----------------------------------------------------------------------------- tokenisation
def encode_state(tok, text: str, max_tokens: int | None = None) -> list[int]:
    ids = tok(text.replace(tok.mask_token, " "), add_special_tokens=False)["input_ids"]
    return ids[:max_tokens] if max_tokens else ids


def encode_block(tok, q: dict, max_opt_tokens: int = 16, max_text_tokens: int = 96):
    """One question block. Returns (ids, marker offsets relative to the block start)."""
    texts = ["choice question: " + q["text"]] + [" " + o for o in q["options"]]
    texts = [t.replace(tok.mask_token, " ") for t in texts]
    enc = tok(texts, add_special_tokens=False)["input_ids"]
    ids = enc[0][:max_text_tokens] + [tok.sep_token_id]
    markers = []
    for opt_ids in enc[1:]:
        markers.append(len(ids))
        ids.append(tok.mask_token_id)
        ids.extend(opt_ids[:max_opt_tokens])
    ids.append(tok.sep_token_id)
    return ids, markers


# ----------------------------------------------------------------------------- packing
def pack_readonce(tok, state_ids: list[int], blocks: list[tuple[list[int], list[int]]],
                  positions: str = "parallel") -> dict:
    """[CLS] state [SEP] + blocks. seg: 0 = state, k = k-th block (1-based)."""
    ids = [tok.cls_token_id] + list(state_ids) + [tok.sep_token_id]
    S = len(ids)
    pos, seg, q_markers = list(range(S)), [0] * S, []
    for k, (bids, bmark) in enumerate(blocks, start=1):
        off = len(ids)
        ids.extend(bids)
        start = S if positions == "parallel" else off  # "sequential" = ordinary 0..L-1 positions
        pos.extend(range(start, start + len(bids)))
        seg.extend([k] * len(bids))
        q_markers.append([off + m for m in bmark])
    return {"ids": ids, "pos": pos, "seg": seg, "q_markers": q_markers, "state_len": S}


def pack_cross(tok, state_ids: list[int], block: tuple[list[int], list[int]]) -> dict:
    """Laya-style row: [CLS] Q block state [SEP]; ordinary positions, full attention (seg all 0)."""
    bids, bmark = block
    ids = [tok.cls_token_id] + list(bids) + list(state_ids) + [tok.sep_token_id]
    return {"ids": ids, "pos": list(range(len(ids))), "seg": [0] * len(ids),
            "q_markers": [[1 + m for m in bmark]], "state_len": None}


def collate(items: list[dict], pad_id: int, labels: list[list[int]] | None = None) -> dict:
    """Pad rows; flatten every question of every row into a [Q, M] table of marker indices
    into the flattened [B*L] hidden states."""
    B, L = len(items), max(len(it["ids"]) for it in items)
    ids = torch.full((B, L), pad_id, dtype=torch.long)
    pos = torch.zeros((B, L), dtype=torch.long)
    seg = torch.full((B, L), -1, dtype=torch.long)
    att = torch.zeros((B, L), dtype=torch.long)
    qm_rows, q_row = [], []
    for b, it in enumerate(items):
        n = len(it["ids"])
        ids[b, :n] = torch.tensor(it["ids"])
        pos[b, :n] = torch.tensor(it["pos"])
        seg[b, :n] = torch.tensor(it["seg"])
        att[b, :n] = 1
        for m in it["q_markers"]:
            qm_rows.append([b * L + x for x in m])
            q_row.append(b)
    Q, M = len(qm_rows), max((len(m) for m in qm_rows), default=1)
    qm = torch.zeros((Q, M), dtype=torch.long)
    qmask = torch.zeros((Q, M), dtype=torch.bool)
    for i, m in enumerate(qm_rows):
        qm[i, :len(m)] = torch.tensor(m)
        qmask[i, :len(m)] = True
    out = {"input_ids": ids, "position_ids": pos, "seg": seg, "attention_mask": att,
           "q_markers": qm, "q_mask": qmask, "q_row": torch.tensor(q_row)}
    if labels is not None:
        out["labels"] = torch.tensor([y for row in labels for y in row], dtype=torch.long)
    return out


# ----------------------------------------------------------------------------- masks
def build_masks(seg: torch.Tensor, pos: torch.Tensor, window: int, impl: str = "sdpa",
                dtype: torch.dtype = torch.float32, window_on: str = "position",
                isolate_blocks: bool = True) -> dict:
    """4D masks [B, 1, L, L] for ModernBertModel's {"full_attention", "sliding_attention"} layers.

    seg: [B, L] (0 = state, k>0 = block k, -1 = padding); pos: [B, L] position ids.
    True = may attend (sdpa); eager gets the additive 0 / finfo.min form.
    """
    B, L = seg.shape
    qs, ks = seg[:, :, None], seg[:, None, :]
    if isolate_blocks:
        allowed = (ks == 0) | (ks == qs)
    else:  # negative control: blocks may see each other (state still sees only state)
        allowed = (ks == 0) | ((qs > 0) & (ks > 0))
    allowed = allowed & (qs >= 0) & (ks >= 0)
    eye = torch.eye(L, dtype=torch.bool, device=seg.device)[None]
    pad_q = (qs < 0) & eye  # padding rows attend to themselves only (avoids all-masked NaN rows)
    allowed = allowed | pad_q
    if window_on == "position":
        dist = (pos[:, :, None] - pos[:, None, :]).abs()
    else:  # what the stock mask does: distance in sequence index
        idx = torch.arange(L, device=seg.device)
        dist = (idx[:, None] - idx[None, :]).abs()[None]
    local = (allowed & (dist <= window)) | pad_q
    masks = {"full_attention": allowed[:, None], "sliding_attention": local[:, None]}
    if impl == "eager":
        neg = torch.finfo(dtype).min
        masks = {k: torch.zeros(v.shape, dtype=dtype).masked_fill(~v, neg) for k, v in masks.items()}
    return masks


# ----------------------------------------------------------------------------- model
class ChoiceModel(nn.Module):
    """Encoder + per-option scorer at each [MASK] marker; softmax over a question's options."""

    def __init__(self, encoder: nn.Module):
        super().__init__()
        self.encoder = encoder
        d = encoder.config.hidden_size
        self.scorer = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, d), nn.GELU(), nn.Linear(d, 1))
        self.window = encoder.config.sliding_window  # half-window, e.g. 64

    def encode(self, batch: dict, mode: str = "readonce", **mask_kw) -> torch.Tensor:
        if mode == "stock":  # plain HF path: 2D padding mask, default positions and masks
            return self.encoder(input_ids=batch["input_ids"],
                                attention_mask=batch["attention_mask"]).last_hidden_state
        impl = self.encoder.config._attn_implementation
        masks = build_masks(batch["seg"], batch["position_ids"], self.window, impl=impl,
                            dtype=next(self.encoder.parameters()).dtype, **mask_kw)
        return self.encoder(input_ids=batch["input_ids"], attention_mask=masks,
                            position_ids=batch["position_ids"]).last_hidden_state

    def score(self, h: torch.Tensor, batch: dict) -> torch.Tensor:
        flat = h.reshape(-1, h.size(-1))
        m = flat[batch["q_markers"]]  # [Q, M, d]
        logits = self.scorer(m).squeeze(-1).float()
        return logits.masked_fill(~batch["q_mask"], float("-inf"))

    def forward(self, batch: dict, mode: str = "readonce", **mask_kw):
        h = self.encode(batch, mode, **mask_kw)
        return self.score(h, batch), h


def choice_loss(logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    return F.cross_entropy(logits, labels)


# ----------------------------------------------------------------------------- convenience
def build_readonce_batch(tok, states: list[str], questions: list[list[dict]], max_state_tokens=None,
                         positions: str = "parallel", with_labels: bool = False) -> dict:
    items = []
    for text, qs in zip(states, questions):
        sids = encode_state(tok, text, max_state_tokens)
        items.append(pack_readonce(tok, sids, [encode_block(tok, q) for q in qs], positions=positions))
    labels = [[q["answer"] for q in qs] for qs in questions] if with_labels else None
    return collate(items, tok.pad_token_id, labels)


def build_cross_batch(tok, states: list[str], questions: list[list[dict]], max_state_tokens=None,
                      with_labels: bool = False) -> dict:
    """One row per (state, question): the state is re-read for every question."""
    items, labels = [], []
    for text, qs in zip(states, questions):
        sids = encode_state(tok, text, max_state_tokens)
        for q in qs:
            items.append(pack_cross(tok, sids, encode_block(tok, q)))
            labels.append([q.get("answer", -1)])
    return collate(items, tok.pad_token_id, labels if with_labels else None)


In [ ]:
%%writefile synth.py
"""Synthetic states and typed questions with deterministic answers.

Two record types, rendered as text:
  * service-metrics record (categorical fields + daily cloud spend, totals, error rate this/last week)
  * support ticket (customer, plan, product, priority, assignee, SLA hours left, age)
Every question is a Choice question (2-30 options); yes/no questions are 2-option choices.
Each question carries a `family`: "lookup" (copy a categorical field) or "numeric"
(threshold / comparison / argmax / bucketing over numbers in the record).

`numeric_style` controls how numbers are rendered (experiment 4):
  raw     - "$1,234.56", "2.70%"
  digits  - digits separated by spaces: "$ 1 2 3 4 . 5 6"
  derived - raw, plus computed facts appended as text (error-rate change, spend vs budget ...)
"""
from __future__ import annotations

import random

COUNTRIES = ["Germany", "France", "Spain", "Italy", "Poland", "Netherlands", "Belgium", "Sweden", "Norway",
             "Denmark", "Finland", "Ireland", "Portugal", "Austria", "Switzerland", "Greece", "Turkey",
             "Brazil", "Mexico", "Canada", "Argentina", "Chile", "Japan", "India", "Australia",
             "Indonesia", "Vietnam", "Egypt", "Nigeria", "Kenya"]
REGIONS = ["eu-west", "eu-north", "us-east", "us-west", "ap-south", "ap-east", "sa-east", "af-south"]
TIERS = ["web", "api", "batch", "streaming", "storage", "analytics"]
STATUSES = ["healthy", "degraded", "maintenance", "scaling", "retired"]
DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
AGENTS = ["Priya", "Marco", "Aisha", "Tomasz", "Chen", "Lucia", "Omar", "Hannah", "Kenji", "Fatima",
          "Diego", "Ingrid", "Ravi", "Sofia", "Kwame", "Elena", "Mateo", "Yuki", "Noah", "Amara",
          "Lars", "Mei", "Jonas", "Leila", "Pedro", "Anika", "Hugo", "Zara", "Felix", "Nadia"]
PLANS = ["Free", "Starter", "Pro", "Business", "Enterprise"]
CHANNELS = ["email", "chat", "phone", "web form", "social"]
PRODUCTS = ["Billing", "Login", "Shipping", "Refunds", "API", "Mobile app", "Reporting", "Integrations"]
PRIORITIES = ["P1", "P2", "P3", "P4"]
SENTIMENTS = ["angry", "frustrated", "neutral", "positive"]
ADJ = ["Bold", "Quiet", "Rapid", "Golden", "Summer", "Winter", "Bright", "Prime", "Urban", "Fresh"]
NOUN = ["Falcon", "Harbor", "Cedar", "Comet", "River", "Wave", "Pulse", "Echo", "Summit", "Orbit"]
SUBJECTS = ["Refund not received for invoice", "Cannot log in after password reset", "Package delayed at depot",
            "API returns 500 on export", "App crashes on startup", "Report totals do not match",
            "Charged twice this month", "Integration token expired"]


# ----------------------------------------------------------------------------- number rendering
def _digits(s: str) -> str:
    return " ".join(ch for ch in s if ch != ",")  # "$1,234.56" -> "$ 1 2 3 4 . 5 6"


def money(x: float, style: str) -> str:
    s = "${:,.2f}".format(x)
    return _digits(s) if style == "digits" else s


def num(x: int, style: str) -> str:
    s = "{:,}".format(x)
    return _digits(s) if style == "digits" else s


def pct_(x: float, style: str) -> str:
    s = "{:.2f}%".format(x)
    return _digits(s) if style == "digits" else s


# ----------------------------------------------------------------------------- records
def service_record(rng: random.Random) -> dict:
    budget = rng.randrange(20, 1001, 10)
    daily = [round(rng.uniform(0.05, 2.0) * budget, 2) for _ in DAYS]
    requests = rng.randint(2_000, 500_000)
    err = round(rng.uniform(0.2, 5.0), 2)
    # last week's error rate differs by at least ~0.1 points, so "is it falling?" is never a tie.
    # (An earlier rejection loop, err * U(0.6, 1.4), never terminated for err < 0.25.)
    delta = rng.uniform(0.1, max(0.15, 0.4 * err))
    err_prev = round(err + delta if (rng.random() < 0.5 or err - delta < 0.05) else err - delta, 2)
    alerts = 0 if rng.random() < 0.5 else rng.randint(1, 200)
    return {"kind": "service", "service": "%s-%s-%d" % (rng.choice(ADJ), rng.choice(NOUN), rng.randint(1, 99)),
            "region": rng.choice(REGIONS), "country": rng.choice(COUNTRIES),
            "tier": rng.choice(TIERS), "status": rng.choice(STATUSES), "budget": budget,
            "daily": daily, "spend": round(sum(daily), 2), "requests": requests,
            "errors": int(round(requests * err / 100)), "alerts": alerts,
            "err": err, "err_prev": err_prev}


def ticket_record(rng: random.Random) -> dict:
    return {"kind": "ticket", "id": rng.randint(10_000, 99_999),
            "customer": "%s %s" % (rng.choice(ADJ), rng.choice(["GmbH", "Ltd", "Inc", "SA", "BV"])),
            "plan": rng.choice(PLANS), "channel": rng.choice(CHANNELS), "product": rng.choice(PRODUCTS),
            "priority": rng.choice(PRIORITIES), "assignee": rng.choice(AGENTS),
            "sentiment": rng.choice(SENTIMENTS), "days_open": rng.randint(0, 30),
            "sla_hours": rng.randint(1, 72), "replies": rng.randint(0, 25),
            "subject": "%s %s-%d" % (rng.choice(SUBJECTS), rng.choice(["INV", "ORD", "REQ"]), rng.randint(100, 9999))}


def render(r: dict, style: str = "raw") -> str:
    if r["kind"] == "service":
        lines = [
            "service: %s" % r["service"],
            "region: %s | country: %s | tier: %s | status: %s" % (r["region"], r["country"],
                                                                 r["tier"], r["status"]),
            "daily budget: %s" % money(r["budget"], style),
            "spend by day: " + ", ".join("%s %s" % (d, money(v, style)) for d, v in zip(DAYS, r["daily"])),
            "total spend: %s | requests: %s | errors: %s | alerts: %s" % (
                money(r["spend"], style), num(r["requests"], style), num(r["errors"], style),
                num(r["alerts"], style)),
            "error rate this week: %s | error rate last week: %s" % (pct_(r["err"], style),
                                                                    pct_(r["err_prev"], style)),
        ]
        if style == "derived":
            d = r["err"] - r["err_prev"]
            top = DAYS[max(range(7), key=lambda i: r["daily"][i])]
            lines.append("derived: error-rate change %+.2f points (%s) | spend is %.1fx the weekly budget | "
                         "highest-spend day %s | zero alerts: %s" % (
                             d, "falling" if d < 0 else "rising", r["spend"] / (7 * r["budget"]), top,
                             "yes" if r["alerts"] == 0 else "no"))
        return "\n".join(lines)
    lines = [
        "ticket #%d from %s (%s plan) via %s" % (r["id"], r["customer"], r["plan"], r["channel"]),
        "subject: %s" % r["subject"],
        "product: %s | priority: %s | assignee: %s | sentiment: %s" % (r["product"], r["priority"],
                                                                      r["assignee"], r["sentiment"]),
        "opened %s days ago | SLA hours left: %s | replies: %s" % (
            num(r["days_open"], style), num(r["sla_hours"], style), num(r["replies"], style)),
    ]
    if style == "derived":
        lines.append("derived: open more than a week: %s | SLA under 12 hours: %s" % (
            "yes" if r["days_open"] > 7 else "no", "yes" if r["sla_hours"] < 12 else "no"))
    return "\n".join(lines)


# ----------------------------------------------------------------------------- questions
def _choice(rng, text, pool, answer, family, template, n_min=2, n_max=None):
    n_max = min(n_max or len(pool), len(pool))
    n = rng.randint(n_min, n_max)
    opts = [answer] + rng.sample([p for p in pool if p != answer], n - 1)
    rng.shuffle(opts)
    return {"text": text, "options": opts, "answer": opts.index(answer), "family": family, "template": template}


def _yesno(rng, text, truth, family, template):
    opts = ["yes", "no"] if rng.random() < 0.5 else ["no", "yes"]
    return {"text": text, "options": opts, "answer": opts.index("yes" if truth else "no"),
            "family": family, "template": template}


def _threshold(rng, value, lo=0.5, hi=1.5, gap=0.05):
    """A threshold that is clearly above or below `value` (50/50)."""
    if rng.random() < 0.5:
        return value * rng.uniform(lo, 1 - gap)
    return value * rng.uniform(1 + gap, hi)


def service_questions(r: dict, rng: random.Random, style: str = "raw") -> list[dict]:
    qs = [
        _choice(rng, "Which region is this service running in?", REGIONS, r["region"], "lookup", "region"),
        _choice(rng, "Which country hosts the service?", COUNTRIES, r["country"], "lookup", "country"),
        _choice(rng, "What is the service status?", STATUSES, r["status"], "lookup", "status"),
        _choice(rng, "What is the service tier?", TIERS, r["tier"], "lookup", "tier"),
    ]
    top = DAYS[max(range(7), key=lambda i: r["daily"][i])]
    qs.append({"text": "On which day was spend highest?", "options": rng.sample(DAYS, 7), "answer": None,
               "family": "numeric", "template": "peak_day"})
    qs[-1]["answer"] = qs[-1]["options"].index(top)
    # spend bucket: n contiguous buckets of equal width, one of which contains total spend
    n = rng.randint(3, 30)
    width = max(50, int(round(r["spend"] / rng.uniform(2, 8) / 50.0)) * 50)
    k = min(int(r["spend"] // width), 10_000)
    first = max(0, k - rng.randint(0, n - 1))
    edges = [(first + i) * width for i in range(n)]
    if not (edges[0] <= r["spend"] < edges[-1] + width):
        first = k
        edges = [(first + i) * width for i in range(n)]
    opts = ["%s to %s" % (money(e, style), money(e + width, style)) for e in edges]
    qs.append({"text": "Which range contains total spend?", "options": opts,
               "answer": [i for i, e in enumerate(edges) if e <= r["spend"] < e + width][0],
               "family": "numeric", "template": "spend_bucket"})
    qs.append(_yesno(rng, "Is the error rate falling compared with last week?", r["err"] < r["err_prev"],
                     "numeric", "err_falling"))
    x = round(_threshold(rng, r["spend"], 0.4, 1.6, 0.08), -1)
    qs.append(_yesno(rng, "Did total spend exceed %s with zero alerts?" % money(x, style),
                     r["spend"] > x and r["alerts"] == 0, "numeric", "spend_zero_alerts"))
    b = round(_threshold(rng, r["budget"], 0.4, 1.6, 0.1))
    qs.append(_yesno(rng, "Is the daily budget above %s?" % money(b, style), r["budget"] > b, "numeric",
                     "budget_above"))
    s = r["status"] if rng.random() < 0.5 else rng.choice(STATUSES)
    qs.append(_yesno(rng, "Is the service %s?" % s, r["status"] == s, "lookup", "status_is"))
    return qs


def ticket_questions(r: dict, rng: random.Random, style: str = "raw") -> list[dict]:
    qs = [
        _choice(rng, "Which product area is this ticket about?", PRODUCTS, r["product"], "lookup", "product"),
        _choice(rng, "Who is the ticket assigned to?", AGENTS, r["assignee"], "lookup", "assignee"),
        _choice(rng, "What is the ticket priority?", PRIORITIES, r["priority"], "lookup", "priority"),
        _choice(rng, "Which plan is the customer on?", PLANS, r["plan"], "lookup", "plan"),
        _choice(rng, "What is the customer's sentiment?", SENTIMENTS, r["sentiment"], "lookup", "sentiment"),
    ]
    p = r["plan"] if rng.random() < 0.5 else rng.choice(PLANS)
    qs.append(_yesno(rng, "Is the customer on the %s plan?" % p, r["plan"] == p, "lookup", "plan_is"))
    h = rng.randint(2, 72)
    while h == r["sla_hours"]:
        h = rng.randint(2, 72)
    qs.append(_yesno(rng, "Are fewer than %s hours left on the SLA?" % num(h, style), r["sla_hours"] < h,
                     "numeric", "sla_under"))
    d = rng.randint(1, 30)
    while d == r["days_open"]:
        d = rng.randint(1, 30)
    qs.append(_yesno(rng, "Has the ticket been open for more than %s days?" % num(d, style),
                     r["days_open"] > d, "numeric", "open_over"))
    return qs


def make_example(rng: random.Random, style: str = "raw") -> dict:
    if rng.random() < 0.5:
        r = service_record(rng)
        qs = service_questions(r, rng, style)
    else:
        r = ticket_record(rng)
        qs = ticket_questions(r, rng, style)
    return {"state": render(r, style), "questions": qs, "kind": r["kind"]}


def make_dataset(n: int, seed: int, style: str = "raw") -> list[dict]:
    rng = random.Random(seed)
    return [make_example(rng, style) for _ in range(n)]


# ----------------------------------------------------------------------------- long states (timing)
def long_state_ids(tok, n_tokens: int, seed: int = 0) -> list[int]:
    """Exactly n_tokens state tokens: concatenated service/ticket records, truncated."""
    rng = random.Random(seed)
    parts, ids = [], []
    while len(ids) < n_tokens:
        for _ in range(8):
            r = service_record(rng) if rng.random() < 0.6 else ticket_record(rng)
            parts.append(render(r))
        ids = tok("\n---\n".join(parts), add_special_tokens=False)["input_ids"]
    return ids[:n_tokens]


def question_pool(k: int, seed: int = 0) -> list[dict]:
    """k questions with a realistic mix of option counts (about half yes/no), for timing/exactness."""
    rng = random.Random(seed)
    out = []
    while len(out) < k:
        ex = make_example(rng)
        out.extend(ex["questions"])
    rng.shuffle(out)
    return out[:k]


In [ ]:
%%writefile smoke.py
"""Kaggle T4 smoke test for the read-once encoder.

Run in a Kaggle notebook with Accelerator = "GPU T4 x2" and Internet on. Each test is
independent: a failure is recorded, not fatal. Results are rewritten to smoke_results.json after
every test, so a session that dies part-way still leaves everything measured so far.

Tests, in priority order:
  env               versions and GPUs
  exactness         read-once packed vs single-question outputs on CUDA, in fp32 and fp16, plus the
                    sequence-index sliding-window negative control
  activations       per-module max |activation| in fp32 (headroom against the fp16 max of 65,504)
                    and a non-finite check in an fp16 forward pass
  train_*           training throughput, peak memory and stability (fp16 AMP; one fp32 and one
                    Laya-style cross-encoder run for comparison)
  sdpa_backends     which SDPA kernels accept the custom mask on this GPU, and their speed
  flex              whether FlexAttention compiles and matches dense SDPA on this GPU
  resume            checkpoint save, reload and continue gives the same loss

Usage on Kaggle:        python smoke.py
Usage for a dry run:    python smoke.py --dry-run    (CPU, tiny shapes; checks the code paths only)
"""
import argparse
import gc
import json
import math
import os
import platform
import subprocess
import sys
import time
import traceback

import torch
import torch.nn.functional as F

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from readonce import (ChoiceModel, choice_loss, collate, encode_block,  # noqa: E402
                      pack_cross, pack_readonce, build_readonce_batch)
from synth import long_state_ids, make_dataset, question_pool  # noqa: E402

# Pinned safetensors revisions (the same ones experiment 01 verified against each repo's main .bin).
MODELS = {
    "ettin-32m": ("jhu-clsp/ettin-encoder-32m", "27b98f697261919d35f3f995fee5c690921e5404"),
    "ettin-68m": ("jhu-clsp/ettin-encoder-68m", "d446589aef55657267b913c9bcce6c98b69061c7"),
    "modernbert-base": ("answerdotai/ModernBERT-base", "8949b909ec900327062f0ebf497f51aef5e6f0c8"),
}
FP16_MAX = 65504.0
OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.dirname(os.path.abspath(__file__))
OUT = os.path.join(OUT_DIR, "smoke_results.json")
RESULTS = {"started": time.strftime("%Y-%m-%d %H:%M:%S"), "tests": {}}


# ----------------------------------------------------------------------------- bookkeeping
def save():
    with open(OUT, "w") as fh:
        json.dump(RESULTS, fh, indent=1, default=str)


def free():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def record(name, fn, deadline):
    if time.time() > deadline:
        RESULTS["tests"][name] = {"ok": False, "skipped": "time budget exhausted"}
        save()
        print(f"[SKIP] {name}", flush=True)
        return
    t0 = time.time()
    try:
        RESULTS["tests"][name] = {"ok": True, "result": fn()}
    except Exception as e:  # noqa: BLE001 - every failure is data here
        RESULTS["tests"][name] = {"ok": False, "error": repr(e)[:500], "trace": traceback.format_exc()[-3000:]}
    RESULTS["tests"][name]["seconds"] = round(time.time() - t0, 1)
    save()
    free()
    t = RESULTS["tests"][name]
    print(f"[{'OK ' if t['ok'] else 'ERR'}] {name} ({t['seconds']} s)", flush=True)
    if not t["ok"]:
        print("      " + t["error"], flush=True)


def sync(device):
    if device.type == "cuda":
        torch.cuda.synchronize(device)


def dev(batch, device):
    return {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in batch.items()}


def median(xs):
    s = sorted(xs)
    return s[len(s) // 2] if s else float("nan")


# ----------------------------------------------------------------------------- models and data
def load(name, device):
    from transformers import AutoModelForMaskedLM, AutoTokenizer
    repo, rev = MODELS[name]
    tok = AutoTokenizer.from_pretrained(repo, revision=rev)
    # Load through the MLM class: these safetensors store the tied input embedding only as
    # decoder.weight, so a bare AutoModel load would leave the embeddings randomly initialised.
    mlm = AutoModelForMaskedLM.from_pretrained(repo, revision=rev, use_safetensors=True,
                                               attn_implementation="sdpa", dtype=torch.float32)
    enc = mlm.model
    if not torch.equal(enc.embeddings.tok_embeddings.weight, mlm.decoder.weight):
        raise RuntimeError("input embedding not tied to decoder.weight for " + name)
    return tok, enc.to(device)


def new_model(enc, device, train):
    torch.manual_seed(0)  # identical scorer initialisation in every test
    m = ChoiceModel(enc).to(device)
    return m.train() if train else m.eval()


def packed_rows(tok, S, K, n_rows, seed):
    """n_rows read-once rows, each with an S-token state block ([CLS] + S-2 + [SEP]) and K questions."""
    items, labels = [], []
    for r in range(n_rows):
        sids = long_state_ids(tok, S - 2, seed=seed + r)
        qs = question_pool(K, seed=seed * 1000 + r)
        items.append(pack_readonce(tok, sids, [encode_block(tok, q) for q in qs]))
        labels.append([q["answer"] for q in qs])
    return collate(items, tok.pad_token_id, labels)


def cross_rows(tok, S, K, n_states, seed):
    """Laya-style rows: one row per (state, question), so each state is re-read K times."""
    items, labels = [], []
    for r in range(n_states):
        sids = long_state_ids(tok, S - 2, seed=seed + r)
        for q in question_pool(K, seed=seed * 1000 + r):
            items.append(pack_cross(tok, sids, encode_block(tok, q)))
            labels.append([q["answer"]])
    return collate(items, tok.pad_token_id, labels)


# ----------------------------------------------------------------------------- tests
def t_env(device):
    import huggingface_hub
    import tokenizers
    import transformers
    info = {"python": sys.version.split()[0], "platform": platform.platform(), "torch": torch.__version__,
            "cuda": torch.version.cuda, "transformers": transformers.__version__,
            "tokenizers": tokenizers.__version__, "huggingface_hub": huggingface_hub.__version__,
            "device": str(device)}
    if torch.cuda.is_available():
        info["gpus"] = [{"name": torch.cuda.get_device_name(i),
                         "capability": list(torch.cuda.get_device_capability(i)),
                         "mem_gb": round(torch.cuda.get_device_properties(i).total_memory / 1e9, 2)}
                        for i in range(torch.cuda.device_count())]
        try:
            info["nvidia_smi"] = subprocess.run(
                ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv"],
                capture_output=True, text=True, timeout=30).stdout
        except Exception as e:  # noqa: BLE001
            info["nvidia_smi"] = repr(e)
    return info


def gpu_state():
    """GPU 0 clocks, temperature, power and active throttle reasons (nvidia-smi), or None off-GPU."""
    if not torch.cuda.is_available():
        return None
    q = "clocks.sm,clocks.max.sm,temperature.gpu,power.draw,clocks_throttle_reasons.active"
    try:
        out = subprocess.run(["nvidia-smi", "-i", "0", f"--query-gpu={q}", "--format=csv,noheader,nounits"],
                             capture_output=True, text=True, timeout=30).stdout.strip()
        return dict(zip(q.split(","), [x.strip() for x in out.split(",")]))
    except Exception as e:  # noqa: BLE001
        return {"error": repr(e)}


def t_exactness(name, device, n_records, long_S, long_K):
    """Each question's probabilities in a packed pass vs the same question packed alone."""
    tok, enc = load(name, device)
    model = new_model(enc, device, train=False)
    cases = [(ex["state"], ex["questions"]) for ex in make_dataset(n_records, seed=123)]
    long_state = tok.decode(long_state_ids(tok, long_S - 2, seed=7))
    cases.append((long_state, question_pool(long_K, seed=7)))
    modes = [("fp32", False, "position"), ("fp16", True, "position"),
             ("fp32_negative_control_index_window", False, "index")]
    out = {}
    for label, half, window_on in modes:
        if half and device.type != "cuda":
            continue
        maxdiff, agree, n = 0.0, 0, 0
        with torch.no_grad(), torch.autocast(device.type, dtype=torch.float16, enabled=half):
            for state, qs in cases:
                lp, _ = model(dev(build_readonce_batch(tok, [state], [qs]), device),
                              mode="readonce", window_on=window_on)
                pp = torch.softmax(lp.float(), -1)
                for j, q in enumerate(qs):
                    ls, _ = model(dev(build_readonce_batch(tok, [state], [[q]]), device), mode="readonce")
                    ps = torch.softmax(ls.float(), -1)
                    k = len(q["options"])
                    maxdiff = max(maxdiff, (pp[j, :k] - ps[0, :k]).abs().max().item())
                    agree += int(pp[j, :k].argmax().item() == ps[0, :k].argmax().item())
                    n += 1
        out[label] = {"max_abs_prob_diff": maxdiff, "argmax_agree": agree, "n_questions": n}
    out["cases"] = {"short_records": n_records, "long_state_tokens": long_S, "long_state_questions": long_K}
    return out


def t_activations(name, device, S, K, n_rows):
    """Largest |activation| per module (fp32) and non-finite values in an fp16 forward pass."""
    tok, enc = load(name, device)
    model = new_model(enc, device, train=False)
    batch = dev(packed_rows(tok, S, K, n_rows, seed=11), device)
    stats = {}

    def mk(nm):
        def hook(_mod, _inp, out):
            t = out[0] if isinstance(out, (tuple, list)) else out
            if torch.is_tensor(t) and t.is_floating_point():
                a = t.detach().abs()
                fin = torch.isfinite(a)
                mx = a[fin].max().item() if fin.any() else float("nan")
                prev = stats.get(nm, {"max": 0.0, "nonfinite": 0})
                stats[nm] = {"max": max(prev["max"], mx), "nonfinite": prev["nonfinite"] + int((~fin).sum().item())}
        return hook

    hooks = []
    for nm, mod in model.encoder.named_modules():
        is_leaf = len(list(mod.children())) == 0
        is_layer = nm.startswith("layers.") and nm.count(".") == 1  # residual stream after each layer
        if nm and (is_leaf or is_layer):
            hooks.append(mod.register_forward_hook(mk(nm)))
    out = {"shape": {"S": S, "K": K, "rows": n_rows, "L": int(batch["input_ids"].shape[1])}}
    try:
        for label, half in (("fp32", False), ("fp16", True)):
            if half and device.type != "cuda":
                continue
            stats.clear()
            with torch.no_grad(), torch.autocast(device.type, dtype=torch.float16, enabled=half):
                logits, _ = model(batch, mode="readonce")
            gmax = max(v["max"] for v in stats.values())
            top = sorted(stats.items(), key=lambda kv: -(kv[1]["max"] if kv[1]["max"] == kv[1]["max"] else 0))[:8]
            out[label] = {"global_max_abs": round(gmax, 1), "headroom_vs_fp16_max": round(FP16_MAX / gmax, 2),
                          "nonfinite_values": sum(v["nonfinite"] for v in stats.values()),
                          "logits_finite": bool(torch.isfinite(logits[batch["q_mask"]]).all().item()),
                          "top_modules": [(k, round(v["max"], 1)) for k, v in top]}
    finally:
        for h in hooks:
            h.remove()
    return out


def t_train(name, device, S, K, bs, steps, amp, arch="readonce", n_batches=4):
    """Median step time, tokens/s, questions/s, peak memory and loss stability over `steps` steps."""
    tok, enc = load(name, device)
    model = new_model(enc, device, train=True)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-5)
    use_amp = amp and device.type == "cuda"
    scaler = torch.amp.GradScaler(device.type, enabled=use_amp)
    if arch == "readonce":
        batches = [dev(packed_rows(tok, S, K, bs, seed=1000 + i), device) for i in range(n_batches)]
    else:  # bs = number of states; rows = bs * K
        batches = [dev(cross_rows(tok, S, K, bs, seed=1000 + i), device) for i in range(n_batches)]
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)
    warm, times, losses, nonfinite = 3, [], [], 0
    try:
        for step in range(steps + warm):
            b = batches[step % n_batches]
            sync(device)
            t0 = time.perf_counter()
            with torch.autocast(device.type, dtype=torch.float16, enabled=use_amp):
                logits, _ = model(b, mode="readonce" if arch == "readonce" else "stock")
                loss = choice_loss(logits, b["labels"])
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sync(device)
            if step >= warm:
                times.append(time.perf_counter() - t0)
            lv = loss.item()
            losses.append(lv)
            nonfinite += int(not math.isfinite(lv))
    except torch.cuda.OutOfMemoryError as e:
        return {"oom": True, "error": repr(e)[:300], "arch": arch, "S": S, "K": K, "bs": bs}
    gpu_after = gpu_state()
    tokens = [int(b["attention_mask"].sum().item()) for b in batches]
    questions = [int(b["labels"].numel()) for b in batches]
    step_s = median(times)
    return {"arch": arch, "amp_fp16": use_amp, "S": S, "K": K, "bs": bs,
            "rows_per_step": int(batches[0]["input_ids"].shape[0]), "L": int(batches[0]["input_ids"].shape[1]),
            "median_step_ms": round(step_s * 1000, 1),
            "tokens_per_s": round(sum(tokens) / len(tokens) / step_s, 1),
            "questions_per_s": round(sum(questions) / len(questions) / step_s, 1),
            "peak_mem_gb": round(torch.cuda.max_memory_allocated(device) / 1e9, 2) if device.type == "cuda" else None,
            "loss_first": round(losses[0], 4), "loss_last": round(losses[-1], 4),
            "nonfinite_losses": nonfinite,
            "grad_scaler_scale_end": scaler.get_scale() if use_amp else None,
            "gpu_after": gpu_after}


def t_sdpa_backends(name, device, S, K, bs):
    """Forward+backward with the custom mask under each SDPA backend (fp16 autocast)."""
    from torch.nn.attention import SDPBackend, sdpa_kernel
    tok, enc = load(name, device)
    model = new_model(enc, device, train=True)
    b = dev(packed_rows(tok, S, K, bs, seed=5), device)
    out = {"S": S, "K": K, "bs": bs}
    for bk in (SDPBackend.EFFICIENT_ATTENTION, SDPBackend.MATH):
        try:
            times = []
            with sdpa_kernel(bk):
                for i in range(6):
                    sync(device)
                    t0 = time.perf_counter()
                    with torch.autocast(device.type, dtype=torch.float16, enabled=device.type == "cuda"):
                        logits, _ = model(b, mode="readonce")
                        loss = choice_loss(logits, b["labels"])
                    loss.backward()
                    model.zero_grad(set_to_none=True)
                    sync(device)
                    if i >= 2:
                        times.append(time.perf_counter() - t0)
            out[bk.name] = {"ok": True, "median_fwd_bwd_ms": round(median(times) * 1000, 1)}
        except Exception as e:  # noqa: BLE001
            out[bk.name] = {"ok": False, "error": repr(e)[:400]}
        free()
    return out


def t_flex(device, S, K, q_len=40, heads=8, head_dim=64):
    """FlexAttention with the read-once block mask: does it compile here, and does it match SDPA?"""
    from torch.nn.attention.flex_attention import create_block_mask, flex_attention
    seg_list = [0] * S
    for k in range(1, K + 1):
        seg_list += [k] * q_len
    seg = torch.tensor(seg_list, device=device)
    L = seg.numel()
    dtype = torch.float16 if device.type == "cuda" else torch.float32
    q, k_, v = (torch.randn(1, heads, L, head_dim, device=device, dtype=dtype) for _ in range(3))
    dense = ((seg[None, :] == 0) | (seg[None, :] == seg[:, None]))[None, None]
    ref = F.scaled_dot_product_attention(q, k_, v, attn_mask=dense)
    res = {"L": L}

    def mask_mod(b, h, qi, ki):
        return (seg[ki] == 0) | (seg[ki] == seg[qi])

    try:
        bm = create_block_mask(mask_mod, B=None, H=None, Q_LEN=L, KV_LEN=L, device=device)
        fn = torch.compile(flex_attention)
        o = fn(q, k_, v, block_mask=bm)
        sync(device)
        res["compiled_ok"] = True
        res["max_abs_diff_vs_sdpa"] = (o.float() - ref.float()).abs().max().item()
        ft, st = [], []
        for _ in range(10):
            sync(device)
            t0 = time.perf_counter()
            fn(q, k_, v, block_mask=bm)
            sync(device)
            ft.append(time.perf_counter() - t0)
            t0 = time.perf_counter()
            F.scaled_dot_product_attention(q, k_, v, attn_mask=dense)
            sync(device)
            st.append(time.perf_counter() - t0)
        res["flex_median_ms"] = round(median(ft) * 1000, 3)
        res["sdpa_dense_median_ms"] = round(median(st) * 1000, 3)
    except Exception as e:  # noqa: BLE001
        res["compiled_ok"] = False
        res["error"] = repr(e)[:600]
    return res


def t_resume(name, device):
    """One step, save, one more step; reload the checkpoint and take the same step again."""
    tok, enc = load(name, device)
    model = new_model(enc, device, train=True)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-5)
    b = dev(packed_rows(tok, 512, 4, 4, seed=21), device)

    def step(m, o):
        with torch.autocast(device.type, dtype=torch.float16, enabled=device.type == "cuda"):
            logits, _ = m(b, mode="readonce")
            loss = choice_loss(logits, b["labels"])
        o.zero_grad(set_to_none=True)
        loss.backward()
        o.step()
        return loss.item()

    step(model, opt)
    path = os.path.join(OUT_DIR, "resume_ckpt.pt")
    torch.save({"model": model.state_dict(), "opt": opt.state_dict()}, path)
    l_continued = step(model, opt)
    _, enc2 = load(name, device)
    model2 = new_model(enc2, device, train=True)
    ck = torch.load(path, map_location=device)  # our own file, written above
    model2.load_state_dict(ck["model"])
    opt2 = torch.optim.AdamW(model2.parameters(), lr=3e-5)
    opt2.load_state_dict(ck["opt"])
    l_resumed = step(model2, opt2)
    size_mb = round(os.path.getsize(path) / 1e6, 1)
    os.remove(path)
    return {"loss_continued": l_continued, "loss_resumed": l_resumed,
            "abs_diff": abs(l_continued - l_resumed), "checkpoint_mb": size_mb}


# ----------------------------------------------------------------------------- main
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--dry-run", action="store_true", help="CPU, tiny shapes: checks code paths only")
    ap.add_argument("--budget-min", type=float, default=50.0, help="skip remaining tests after this")
    ap.add_argument("--throughput-reps", type=int, default=0,
                    help="if > 0: only run the read-once vs per-question training-throughput pair this many "
                         "times, alternating which goes first, then stop")
    args = ap.parse_args()
    device = torch.device("cuda:0" if torch.cuda.is_available() and not args.dry_run else "cpu")
    deadline = time.time() + args.budget_min * 60
    RESULTS["mode"] = "dry-run" if args.dry_run else "full"
    d = args.dry_run

    record("env", lambda: t_env(device), deadline)
    if args.throughput_reps > 0:
        pair = [("readonce", 2 if d else 16), ("cross", 2 if d else 4)]  # Ettin-68m, K=8, fp16 on GPU
        S_tp, steps_tp = (128, 2) if d else (512, 30)
        for r in range(args.throughput_reps):
            for arch, bs in (pair if r % 2 == 0 else pair[::-1]):
                record(f"tp_r{r}_{arch}", lambda arch=arch, bs=bs:
                       t_train("ettin-68m", device, S_tp, 8, bs, steps_tp, True, arch), deadline)
        RESULTS["finished"] = time.strftime("%Y-%m-%d %H:%M:%S")
        save()
        return
    record("exactness_ettin-68m", lambda: t_exactness("ettin-68m", device, n_records=2 if d else 6,
                                                      long_S=256 if d else 2048, long_K=3 if d else 10), deadline)
    act_models = ["ettin-68m"] if d else ["ettin-68m", "modernbert-base", "ettin-32m"]
    for nm in act_models:
        record(f"activations_{nm}", lambda nm=nm: t_activations(nm, device, S=256 if d else 2048,
                                                                 K=2 if d else 8, n_rows=1 if d else 2), deadline)

    steps = 2 if d else 20
    if d:
        train_cfgs = [("ettin-68m", 128, 2, 2, True, "readonce")]
    else:
        train_cfgs = [("ettin-68m", 512, 8, 16, True, "readonce"),
                      ("ettin-68m", 2048, 8, 4, True, "readonce"),
                      ("ettin-32m", 512, 8, 16, True, "readonce"),
                      ("ettin-32m", 2048, 8, 4, True, "readonce"),
                      ("modernbert-base", 512, 8, 8, True, "readonce"),
                      ("modernbert-base", 2048, 8, 2, True, "readonce"),
                      ("ettin-68m", 512, 8, 16, False, "readonce"),   # fp32 comparison
                      ("ettin-68m", 512, 8, 4, True, "cross")]        # Laya-style: 4 states x 8 questions
    for nm, S, K, bs, amp, arch in train_cfgs:
        label = f"train_{arch}_{nm}_S{S}_K{K}_bs{bs}_{'fp16' if amp else 'fp32'}"
        record(label, lambda nm=nm, S=S, K=K, bs=bs, amp=amp, arch=arch:
               t_train(nm, device, S, K, bs, steps, amp, arch), deadline)

    record("sdpa_backends_ettin-68m", lambda: t_sdpa_backends("ettin-68m", device, S=256 if d else 2048,
                                                              K=2 if d else 8, bs=1 if d else 2), deadline)
    record("flex", lambda: t_flex(device, S=256 if d else 2048, K=2 if d else 8), deadline)
    record("resume_ettin-32m", lambda: t_resume("ettin-68m" if d else "ettin-32m", device), deadline)

    RESULTS["finished"] = time.strftime("%Y-%m-%d %H:%M:%S")
    save()
    ok = sum(1 for t in RESULTS["tests"].values() if t.get("ok"))
    print(f"\nDone: {ok}/{len(RESULTS['tests'])} tests OK. Results: {OUT}", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
!python smoke.py --budget-min 50

In [ ]:
import json
r = json.load(open("/kaggle/working/smoke_results.json"))
for name, t in r["tests"].items():
    if not t.get("ok"):
        print(f"ERR  {name}: {t.get('error') or t.get('skipped')}")
        continue
    res = t["result"]
    if name.startswith("train_"):
        keep = ("oom", "median_step_ms", "tokens_per_s", "questions_per_s", "peak_mem_gb", "loss_first", "loss_last", "nonfinite_losses", "grad_scaler_scale_end")
    elif name.startswith("activations_"):
        res = {k: {kk: v[kk] for kk in ("global_max_abs", "headroom_vs_fp16_max", "nonfinite_values", "logits_finite")} for k, v in res.items() if k in ("fp32", "fp16")}
        keep = None
    elif name == "env":
        keep = ("torch", "cuda", "transformers", "gpus")
    else:
        keep = None
    if keep:
        res = {k: res[k] for k in keep if k in res}
    print(f"OK   {name} ({t['seconds']} s): {json.dumps(res)[:700]}")
